# Predição de UTI pediátrica em casos de SRAG-PE

Desenvolvi este trabalho na disciplina de Aprendizado de Máquinas e Ciência de Dados, em grupo de cinco pessoas. Minha contribuição foi a limpeza de leakage, o tratamento de nulos, a execução do baseline e a etapa de Árvore de Decisão.

Apresento os resultados obtidos no [README](../../README.md) e em [execução e resultados](../../docs/execucao.md). Descrevo a base em [dados](../../docs/dataset.md). Mantenho este notebook sem saídas e salvo os arquivos gerados em `results/`.


## Configuração inicial

Concentro nesta seção as importações, constantes e caminhos utilizados na análise.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    make_scorer,
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
    ConfusionMatrixDisplay
)
from sklearn.base import clone

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SEED = 42

# Localiza a raiz a partir da pasta de trabalho do kernel.
pasta_atual = Path.cwd().resolve()
RAIZ_PROJETO = next(
    (p for p in [pasta_atual, *pasta_atual.parents]
     if (p / "data" / "README.md").is_file()
     and (p / "requirements.txt").is_file()),
    None
)
if RAIZ_PROJETO is None:
    raise RuntimeError("Execute o notebook a partir da raiz do projeto ou de uma subpasta.")

PASTA_DADOS = RAIZ_PROJETO / "data"
PASTA_PROCESSADOS = PASTA_DADOS / "processed"
PASTA_RESULTADOS = RAIZ_PROJETO / "results"
PASTA_TABELAS = PASTA_RESULTADOS / "tables"
PASTA_RELATORIOS = PASTA_RESULTADOS / "reports"
PASTA_FIGURAS = PASTA_RESULTADOS / "figures"
for pasta in [PASTA_PROCESSADOS, PASTA_TABELAS, PASTA_RELATORIOS, PASTA_FIGURAS]:
    pasta.mkdir(parents=True, exist_ok=True)
CAMINHO_BASE = PASTA_DADOS / "raw" / "srag_pediatrico_filtrado.csv"
CAMINHO_RESULTADOS_E2 = PASTA_DADOS / "reference" / "resultados_pipeline_completo.csv"
EXPORTAR_BASE_LIMPA = False


## Contextualização e objetivo

Na Entrega 3, comparamos Regressão Logística, Naive Bayes, Árvore de Decisão, SVM e MLP para classificar o registro de UTI pediátrica (`UTI_BIN`) em casos de SRAG-PE.

Mantemos o pipeline da Entrega 2 e variamos o estimador final:

imputação/encoding → SelectKBest(chi2) → SMOTE → RobustScaler → modelo

Posicionamos o chi-quadrado antes do escalonamento porque ele exige valores não negativos. Aplicamos SMOTE dentro do pipeline, apenas no treino de cada fold. Priorizamos F1, recall e precisão, considerando a raridade da classe positiva.

## Carregamento e limpeza da base


### Carregamento do dataset


In [ ]:
if not Path(CAMINHO_BASE).exists():
    raise FileNotFoundError(
        f"Arquivo '{CAMINHO_BASE}' não encontrado. "
        "Coloque o CSV em data/raw/. Consulte data/README.md para origem e formato esperado."
    )

df_raw = pd.read_csv(CAMINHO_BASE, low_memory=False)

print("Base carregada com sucesso.")
print("Shape inicial:", df_raw.shape)

### Tratamento de inconsistências identificadas na Entrega 2


In [ ]:
colunas_necessarias_ruido = ["UTI", "SUPORT_VEN", "UTI_BIN", "EVOLUCAO"]
faltantes = [col for col in colunas_necessarias_ruido if col not in df_raw.columns]

if faltantes:
    raise ValueError(f"Colunas necessárias para remoção de ruído ausentes: {faltantes}")

ruido_tipo1 = ((df_raw["UTI"] == 2.0) & (df_raw["SUPORT_VEN"] == 1.0))
ruido_tipo2 = ((df_raw["UTI_BIN"] == 0) & (df_raw["EVOLUCAO"] == 2.0))

mask_ruido = ruido_tipo1 | ruido_tipo2

print("Ruído tipo 1 — intubados sem UTI:", int(ruido_tipo1.sum()))
print("Ruído tipo 2 — óbitos sem UTI:", int(ruido_tipo2.sum()))
print("Total de registros removidos:", int(mask_ruido.sum()))

df = df_raw.loc[~mask_ruido].copy()

print("Shape após remoção:", df.shape)

### Seleção das variáveis oficiais da Entrega 2


In [ ]:
FEATURES_SELECIONADAS = [
    "IDADE_ANOS",
    "CS_SEXO",
    "REGIAO",

    "FEBRE",
    "TOSSE",
    "GARGANTA",
    "DISPNEIA",
    "DESC_RESP",
    "SATURACAO",
    "DIARREIA",
    "VOMITO",

    "CARDIOPATI",
    "ASMA",
    "DIABETES",
    "NEUROLOGIC",
    "SIND_DOWN",
    "HEMATOLOGI",
    "HEPATICA",
    "PNEUMOPATI",
    "IMUNODEPRE",
    "RENAL",
    "OBESIDADE",

    "UTI_BIN"
]

faltantes = [col for col in FEATURES_SELECIONADAS if col not in df.columns]

if faltantes:
    raise ValueError(f"Colunas esperadas ausentes no dataset: {faltantes}")

df = df[FEATURES_SELECIONADAS].copy()

print("Shape após seleção de features:", df.shape)
print("Quantidade de features:", df.shape[1] - 1)

### Recodificações fixas


In [ ]:
COMORBIDADES = [
    "CARDIOPATI",
    "ASMA",
    "DIABETES",
    "NEUROLOGIC",
    "SIND_DOWN",
    "HEMATOLOGI",
    "HEPATICA",
    "PNEUMOPATI",
    "IMUNODEPRE",
    "RENAL",
    "OBESIDADE"
]

SINTOMAS = [
    "FEBRE",
    "TOSSE",
    "GARGANTA",
    "DISPNEIA",
    "DESC_RESP",
    "SATURACAO",
    "DIARREIA",
    "VOMITO"
]

# Recodifica comorbidades como variáveis binárias
for col in COMORBIDADES:
    df[col] = (df[col] == 1.0).astype(int)

# Recodifica sintomas preservando ausentes para imputação
for col in SINTOMAS:
    df[col] = df[col].replace({
        1.0: 1,
        2.0: 0,
        9.0: np.nan
    })

# Recodifica sexo como indicador binário
df["IS_FEMALE"] = df["CS_SEXO"].map({
    "F": 1,
    "M": 0
}).astype(float)

df = df.drop(columns=["CS_SEXO"])

print("Recodificação concluída.")
print("Shape final da base:", df.shape)

nans = df.isna().sum()
print("\nColunas com valores ausentes:")
display(nans[nans > 0].sort_values(ascending=False))


### Definição de X e y


In [ ]:
X = df.drop(columns=["UTI_BIN"]).copy()
y = df["UTI_BIN"].astype(int).copy()

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

print("\nColunas de X:")
print(list(X.columns))

print("\nDistribuição da variável alvo:")
display(y.value_counts().sort_index().rename({0: "Sem UTI", 1: "Com UTI"}))

## Distribuição da variável alvo

Observo que a classe positiva é minoritária. Por isso, interpreto F1, recall e precisão junto da acurácia.

In [ ]:
distribuicao = y.value_counts().sort_index()

tabela_distribuicao = pd.DataFrame({
    "Classe": ["Sem UTI", "Com UTI"],
    "Quantidade": [
        int(distribuicao.get(0, 0)),
        int(distribuicao.get(1, 0))
    ],
    "Percentual (%)": [
        round(distribuicao.get(0, 0) / len(y) * 100, 2),
        round(distribuicao.get(1, 0) / len(y) * 100, 2)
    ]
})

display(tabela_distribuicao)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tabela_distribuicao["Classe"], tabela_distribuicao["Quantidade"])
ax.set_title("Distribuição da variável alvo (UTI_BIN)")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade de registros")

for i, valor in enumerate(tabela_distribuicao["Quantidade"]):
    ax.text(i, valor, str(valor), ha="center", va="bottom")

plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "distribuicao_alvo.png", dpi=150, bbox_inches="tight")
plt.show()

razao = distribuicao.get(0, 0) / distribuicao.get(1, 1)
print(f"Razão aproximada entre as classes: {razao:.1f}:1")


## Divisão treino/teste estratificada

Divido os dados após a limpeza e as recodificações fixas. Utilizo `stratify=y` para preservar aproximadamente a proporção das classes.

In [ ]:
# Divide a base mantendo a proporção das classes
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print("Formato X_treino:", X_treino.shape)
print("Formato X_teste:", X_teste.shape)
print("Formato y_treino:", y_treino.shape)
print("Formato y_teste:", y_teste.shape)

print("\nDistribuição da classe no conjunto completo:")
display(y.value_counts(normalize=True).sort_index())

print("\nDistribuição da classe no treino:")
display(y_treino.value_counts(normalize=True).sort_index())

print("\nDistribuição da classe no teste:")
display(y_teste.value_counts(normalize=True).sort_index())


### Validações da base antes da modelagem


In [ ]:
colunas_texto = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Colunas textuais em X:", colunas_texto)

colunas_texto_esperadas = ["REGIAO"]
colunas_texto_inesperadas = [
    col for col in colunas_texto
    if col not in colunas_texto_esperadas
]

if colunas_texto_inesperadas:
    raise ValueError(
        f"Existem colunas textuais inesperadas em X: {colunas_texto_inesperadas}. "
        "Remova ou trate essas colunas antes do pipeline."
    )

if y.isna().sum() > 0:
    raise ValueError("A variável alvo y contém valores ausentes.")

print("Validação da base concluída.")

## Baseline e referência da Entrega 2

Mantenho nesta tabela o baseline e o KNN da Entrega 2 como referências históricas. Não os interpreto como experimentos recalculados no treino da Entrega 3.

In [ ]:
def carregar_referencias_entrega2(caminho_csv=CAMINHO_RESULTADOS_E2):
    """Carrega as referências históricas da Entrega 2."""

    if Path(caminho_csv).exists():
        resultados_e2 = pd.read_csv(caminho_csv)

        baseline = resultados_e2[resultados_e2["imputer"] == "BASELINE"].iloc[0]
        melhor = resultados_e2.sort_values("f1_mean", ascending=False).iloc[0]

        referencias = pd.DataFrame([
            {
                "nome": "Baseline — classe majoritária",
                "f1_mean": baseline["f1_mean"],
                "f1_std": baseline["f1_std"],
                "recall_mean": baseline["recall_mean"],
                "recall_std": baseline["recall_std"],
                "precisao_mean": baseline["precision_mean"],
                "precisao_std": baseline["precision_std"],
                "acuracia_mean": baseline["accuracy_mean"],
                "acuracia_std": baseline["accuracy_std"],
                "best_params": "-"
            },
            {
                "nome": "KNN — referência Entrega 2",
                "f1_mean": melhor["f1_mean"],
                "f1_std": melhor["f1_std"],
                "recall_mean": melhor["recall_mean"],
                "recall_std": melhor["recall_std"],
                "precisao_mean": melhor["precision_mean"],
                "precisao_std": melhor["precision_std"],
                "acuracia_mean": melhor["accuracy_mean"],
                "acuracia_std": melhor["accuracy_std"],
                "best_params": {
                    "imputer": melhor["imputer"],
                    "encoder": melhor["encoder"],
                    "scaler": melhor["scaler"],
                    "balancer": melhor["balancer"],
                    "selector": melhor["selector"],
                }
            }
        ])

        return referencias

    referencias = pd.DataFrame([
        {
            "nome": "Baseline — classe majoritária",
            "f1_mean": 0.0000,
            "f1_std": 0.0000,
            "recall_mean": 0.0000,
            "recall_std": 0.0000,
            "precisao_mean": 0.0000,
            "precisao_std": 0.0000,
            "acuracia_mean": 0.9498,
            "acuracia_std": 0.0012,
            "best_params": "-"
        },
        {
            "nome": "KNN — referência Entrega 2",
            "f1_mean": 0.1494,
            "f1_std": 0.0913,
            "recall_mean": 0.2119,
            "recall_std": 0.1359,
            "precisao_mean": 0.1156,
            "precisao_std": 0.0685,
            "acuracia_mean": 0.8829,
            "acuracia_std": 0.0064,
            "best_params": {
                "imputer": "KNNImputer_k5",
                "encoder": "OneHotEncoder",
                "scaler": "RobustScaler",
                "balancer": "SMOTE",
                "selector": "SelectKBest_chi2",
            }
        }
    ])

    return referencias


referencias_entrega2 = carregar_referencias_entrega2()

display(referencias_entrega2)


### Exportação opcional da base limpa


In [ ]:
if EXPORTAR_BASE_LIMPA:
    caminho_base_limpa = PASTA_PROCESSADOS / "srag_pediatrico_limpo.csv"
    df.to_csv(caminho_base_limpa, index=False)
    print(f"Base limpa exportada localmente: {caminho_base_limpa}")
else:
    print("Exportação de dados por registro desativada.")


## Pipeline oficial de pré-processamento


### Definição dos grupos de variáveis


In [ ]:
COLS_CATEGORICAS = [col for col in ["REGIAO"] if col in X.columns]
COLS_NUMERICAS = [col for col in ["IDADE_ANOS"] if col in X.columns]

COLS_BINARIAS = [
    col for col in X.columns
    if col not in COLS_CATEGORICAS + COLS_NUMERICAS
]

print("Colunas categóricas:", COLS_CATEGORICAS)
print("Colunas numéricas:", COLS_NUMERICAS)
print("Quantidade de colunas binárias:", len(COLS_BINARIAS))

### Compatibilidade com versões do scikit-learn


In [ ]:
def criar_onehot_encoder():
    """Cria encoder compatível com diferentes versões do scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


### Construção do pipeline oficial


In [ ]:
def construir_pipeline_oficial(estimador):
    """Monta o pipeline oficial reaproveitado da Entrega 2."""

    transformadores = []

    if COLS_NUMERICAS:
        transformadores.append(
            (
                "num",
                SkPipeline(steps=[
                    ("imputer", KNNImputer(n_neighbors=5))
                ]),
                COLS_NUMERICAS
            )
        )

    if COLS_CATEGORICAS:
        transformadores.append(
            (
                "cat",
                SkPipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", criar_onehot_encoder())
                ]),
                COLS_CATEGORICAS
            )
        )

    if COLS_BINARIAS:
        transformadores.append(
            (
                "bin",
                SkPipeline(steps=[
                    ("imputer", KNNImputer(n_neighbors=5))
                ]),
                COLS_BINARIAS
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformadores,
        remainder="drop"
    )

    pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("selector", SelectKBest(score_func=chi2, k=10)),
        ("smote", SMOTE(random_state=SEED)),
        ("scaler", RobustScaler()),
        ("estimador", estimador)
    ])

    return pipeline


def construir_pipeline(estimador):
    """Alias único para compatibilidade com chamadas antigas."""
    return construir_pipeline_oficial(estimador)


print("Pipeline oficial definido.")


## Estratégia de validação e busca de hiperparâmetros

Realizamos a validação cruzada de cinco folds e a busca no conjunto de treino. Reservamos o teste para a avaliação após a busca.

In [ ]:
N_FOLDS = 5

cv_5fold = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

METRICAS = {
    "f1": make_scorer(f1_score, pos_label=1, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "precisao": make_scorer(precision_score, pos_label=1, zero_division=0),
    "acuracia": make_scorer(accuracy_score)
}

print("Validação cruzada:", cv_5fold)
print("Métricas:", list(METRICAS.keys()))

### Verificação da ordem do pipeline


In [ ]:
pipeline_teste = construir_pipeline_oficial(
    DummyClassifier(strategy="most_frequent")
)

etapas_oficiais = ["preprocessor", "selector", "smote", "scaler", "estimador"]
etapas_pipeline = [nome for nome, _ in pipeline_teste.steps]

if etapas_pipeline != etapas_oficiais:
    raise AssertionError(f"Ordem incorreta do pipeline: {etapas_pipeline}")

print("Etapas do pipeline oficial:")
print(etapas_pipeline)


### Avaliação por validação cruzada


In [ ]:
def rodar_experimento(nome, estimador, verbose=True):
    """Avalia um estimador fixo com o pipeline oficial."""

    pipeline = construir_pipeline_oficial(estimador)

    inicio = time.time()

    scores = cross_validate(
        estimator=pipeline,
        X=X_treino,
        y=y_treino,
        cv=cv_5fold,
        scoring=METRICAS,
        return_train_score=False,
        n_jobs=-1
    )

    tempo = time.time() - inicio

    resultado = {
        "nome": nome,
        "f1_mean": round(float(np.mean(scores["test_f1"])), 4),
        "f1_std": round(float(np.std(scores["test_f1"])), 4),
        "recall_mean": round(float(np.mean(scores["test_recall"])), 4),
        "recall_std": round(float(np.std(scores["test_recall"])), 4),
        "precisao_mean": round(float(np.mean(scores["test_precisao"])), 4),
        "precisao_std": round(float(np.std(scores["test_precisao"])), 4),
        "acuracia_mean": round(float(np.mean(scores["test_acuracia"])), 4),
        "acuracia_std": round(float(np.std(scores["test_acuracia"])), 4),
        "best_params": "-",
        "tempo_s": round(tempo, 2)
    }

    if verbose:
        print(f"\n{nome}")
        print("-" * 60)
        print(f"F1:       {resultado['f1_mean']:.4f} ± {resultado['f1_std']:.4f}")
        print(f"Recall:   {resultado['recall_mean']:.4f} ± {resultado['recall_std']:.4f}")
        print(f"Precisão: {resultado['precisao_mean']:.4f} ± {resultado['precisao_std']:.4f}")
        print(f"Acurácia: {resultado['acuracia_mean']:.4f} ± {resultado['acuracia_std']:.4f}")
        print(f"Tempo:    {resultado['tempo_s']}s")

    return resultado


### Função de busca aleatória de hiperparâmetros


In [ ]:
def _instanciar_estimador(estimador_ou_classe):
    """Cria uma instância do estimador informado."""

    if isinstance(estimador_ou_classe, type):
        estimador = estimador_ou_classe()
    else:
        estimador = clone(estimador_ou_classe)

    if hasattr(estimador, "get_params") and "random_state" in estimador.get_params():
        estimador.set_params(random_state=SEED)

    return estimador


def _prefixar_parametros_estimador(espaco_params):
    """Adiciona o prefixo estimador__ aos hiperparâmetros."""

    params_prefixados = {}

    for chave, valor in espaco_params.items():
        if "__" in chave:
            params_prefixados[chave] = valor
        else:
            params_prefixados[f"estimador__{chave}"] = valor

    return params_prefixados


def busca_aleatoria(
    nome_modelo,
    estimador_ou_classe,
    espaco_params,
    n_iter=50,
    verbose=True
):
    """Seleciona hiperparâmetros por F1-score no conjunto de treino."""

    estimador = _instanciar_estimador(estimador_ou_classe)
    pipeline = construir_pipeline_oficial(estimador)

    params = _prefixar_parametros_estimador(espaco_params)

    busca = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=n_iter,
        scoring=METRICAS,
        refit="f1",
        cv=cv_5fold,
        random_state=SEED,
        n_jobs=-1,
        return_train_score=False,
        verbose=1 if verbose else 0,
        error_score=np.nan
    )

    inicio = time.time()
    busca.fit(X_treino, y_treino)
    tempo = time.time() - inicio

    idx = busca.best_index_
    cv_results = busca.cv_results_

    melhor_resultado = {
        "nome": f"{nome_modelo} — melhor",
        "f1_mean": round(float(cv_results["mean_test_f1"][idx]), 4),
        "f1_std": round(float(cv_results["std_test_f1"][idx]), 4),
        "recall_mean": round(float(cv_results["mean_test_recall"][idx]), 4),
        "recall_std": round(float(cv_results["std_test_recall"][idx]), 4),
        "precisao_mean": round(float(cv_results["mean_test_precisao"][idx]), 4),
        "precisao_std": round(float(cv_results["std_test_precisao"][idx]), 4),
        "acuracia_mean": round(float(cv_results["mean_test_acuracia"][idx]), 4),
        "acuracia_std": round(float(cv_results["std_test_acuracia"][idx]), 4),
        "best_params": busca.best_params_,
        "tempo_s": round(tempo, 2)
    }

    if verbose:
        print(f"\nMelhor resultado — {nome_modelo}")
        print("-" * 60)
        print("Melhores hiperparâmetros:")
        print(busca.best_params_)
        print()
        print(f"F1:       {melhor_resultado['f1_mean']:.4f} ± {melhor_resultado['f1_std']:.4f}")
        print(f"Recall:   {melhor_resultado['recall_mean']:.4f} ± {melhor_resultado['recall_std']:.4f}")
        print(f"Precisão: {melhor_resultado['precisao_mean']:.4f} ± {melhor_resultado['precisao_std']:.4f}")
        print(f"Acurácia: {melhor_resultado['acuracia_mean']:.4f} ± {melhor_resultado['acuracia_std']:.4f}")
        print(f"Tempo:    {melhor_resultado['tempo_s']}s")

    return melhor_resultado, busca


### Tabela de validação cruzada no conjunto de treino


In [ ]:
TABELA_RESULTADOS = referencias_entrega2.copy()

def adicionar_resultado(resultado):
    """Adiciona um resultado à tabela global."""

    global TABELA_RESULTADOS

    TABELA_RESULTADOS = pd.concat(
        [TABELA_RESULTADOS, pd.DataFrame([resultado])],
        ignore_index=True
    )

    print(f"Resultado adicionado: {resultado['nome']}")


def exibir_tabela_comparacao():
    """Exibe resultados de validação cruzada ordenados por F1 médio."""

    tabela = TABELA_RESULTADOS.sort_values(
        by="f1_mean",
        ascending=False
    ).reset_index(drop=True)

    colunas_exibir = [
        "nome",
        "f1_mean",
        "f1_std",
        "recall_mean",
        "recall_std",
        "precisao_mean",
        "precisao_std",
        "acuracia_mean",
        "acuracia_std"
    ]

    print("Tabela de validação cruzada no conjunto de treino")
    display(
        tabela[colunas_exibir].rename(columns={
            "nome": "Modelo",
            "f1_mean": "F1 médio",
            "f1_std": "F1 std",
            "recall_mean": "Recall médio",
            "recall_std": "Recall std",
            "precisao_mean": "Precisão média",
            "precisao_std": "Precisão std",
            "acuracia_mean": "Acurácia média",
            "acuracia_std": "Acurácia std"
        })
    )

    return tabela


print("Tabela global inicializada com referências da Entrega 2.")
display(TABELA_RESULTADOS)


### Função de avaliação no conjunto reservado de teste


In [ ]:
RESULTADOS_TESTE_FINAL = []

def avaliar_no_teste_final(nome_modelo, modelo_ou_busca):
    """Avalia um modelo já treinado no teste final."""

    if hasattr(modelo_ou_busca, "best_estimator_"):
        modelo_final = modelo_ou_busca.best_estimator_
        best_params = modelo_ou_busca.best_params_
    else:
        modelo_final = modelo_ou_busca
        best_params = "-"

    y_pred = modelo_final.predict(X_teste)

    resultado = {
        "Modelo": nome_modelo,
        "F1 teste": round(f1_score(y_teste, y_pred, zero_division=0), 4),
        "Recall teste": round(recall_score(y_teste, y_pred, zero_division=0), 4),
        "Precisão teste": round(precision_score(y_teste, y_pred, zero_division=0), 4),
        "Acurácia teste": round(accuracy_score(y_teste, y_pred), 4),
        "best_params": best_params
    }

    RESULTADOS_TESTE_FINAL.append(resultado)

    print(f"\nResultado no teste final — {nome_modelo}")
    print("-" * 60)
    print(f"F1:       {resultado['F1 teste']:.4f}")
    print(f"Recall:   {resultado['Recall teste']:.4f}")
    print(f"Precisão: {resultado['Precisão teste']:.4f}")
    print(f"Acurácia: {resultado['Acurácia teste']:.4f}")

    return resultado


def exibir_tabela_teste_final():
    tabela = pd.DataFrame(RESULTADOS_TESTE_FINAL)

    tabela = tabela.sort_values(
        by="F1 teste",
        ascending=False
    ).reset_index(drop=True)

    print("Tabela de avaliação no conjunto reservado de teste")
    display(tabela[[
        "Modelo",
        "F1 teste",
        "Recall teste",
        "Precisão teste",
        "Acurácia teste"
    ]])

    return tabela


### Funções auxiliares disponíveis


In [ ]:
print("Funções disponíveis:")
print("- construir_pipeline_oficial(estimador)")
print("- construir_pipeline(estimador)")
print("- rodar_experimento(nome, estimador)")
print("- busca_aleatoria(nome_modelo, estimador_ou_classe, espaco_params, n_iter)")
print("- adicionar_resultado(resultado)")
print("- exibir_tabela_comparacao()")

Nas seções seguintes, comparamos estimadores com o mesmo pipeline. Selecionamos os hiperparâmetros por F1 no treino.

## Regressão Logística e Naive Bayes


### Importação dos estimadores


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import numpy as np

## Regressão Logística


In [ ]:
parametros_logistica = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["liblinear", "saga"],
    "class_weight": [None, "balanced"],
    "max_iter": [500, 1000]
}

resultado_logistica, busca_logistica = busca_aleatoria(
    nome_modelo="Regressão Logística",
    estimador_ou_classe=LogisticRegression(),
    espaco_params=parametros_logistica,
    n_iter=50,
    verbose=True
)

adicionar_resultado(resultado_logistica)

## Naive Bayes


In [ ]:
parametros_naive_bayes = {
    "var_smoothing": np.logspace(-12, -6, 10)
}

resultado_naive_bayes, busca_naive_bayes = busca_aleatoria(
    nome_modelo="Naive Bayes — GaussianNB",
    estimador_ou_classe=GaussianNB(),
    espaco_params=parametros_naive_bayes,
    n_iter=50,
    verbose=True
)

adicionar_resultado(resultado_naive_bayes)

### Comparação parcial por validação cruzada


In [ ]:
print("Resultado — Regressão Logística")
print(resultado_logistica)

print("\nResultado — Naive Bayes")
print(resultado_naive_bayes)

print("\nComparação parcial após Regressão Logística e Naive Bayes:")
tabela_parcial_logistica_nb = exibir_tabela_comparacao()


## Árvore de Decisão


### Importação do estimador


In [ ]:
from sklearn.tree import DecisionTreeClassifier

### Busca de hiperparâmetros: Árvore de Decisão


In [ ]:
parametros_arvore = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

resultado_arvore, busca_arvore = busca_aleatoria(
    nome_modelo="Árvore de Decisão",
    estimador_ou_classe=DecisionTreeClassifier(random_state=SEED),
    espaco_params=parametros_arvore,
    n_iter=50,
    verbose=True
)

adicionar_resultado(resultado_arvore)

### Comparação parcial por validação cruzada


In [ ]:
tabela_parcial_arvore = exibir_tabela_comparacao()


### Melhor configuração da Árvore de Decisão


In [ ]:
print("Melhor resultado da Árvore de Decisão:")
print(resultado_arvore)

print("\nMelhores hiperparâmetros encontrados:")
print(busca_arvore.best_params_)

## Support Vector Machine


In [ ]:
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
TABELA_RESULTADOS = TABELA_RESULTADOS[
    ~TABELA_RESULTADOS["nome"].isin([
        "SVM (linear) — melhor",
        "SVM (rbf) — melhor",
        "Rede Neural — MLP — melhor"
    ])
].reset_index(drop=True)

### Busca de hiperparâmetros — SVM


In [ ]:
parametros_svm_linear = {
    "C": [0.01, 0.1, 1, 10, 100],
    "class_weight": [None, "balanced"]
}

resultado_svm_linear, busca_svm_linear = busca_aleatoria(
    nome_modelo="SVM Linear",
    estimador_ou_classe=SVC(kernel="linear", random_state=SEED),
    espaco_params=parametros_svm_linear,
    n_iter=10,
    verbose=True
)

# Define grade de busca para SVM RBF
parametros_svm_rbf = {
    "C": [0.01, 0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1, 1],
    "class_weight": [None, "balanced"]
}

resultado_svm_rbf, busca_svm_rbf = busca_aleatoria(
    nome_modelo="SVM RBF",
    estimador_ou_classe=SVC(kernel="rbf", random_state=SEED),
    espaco_params=parametros_svm_rbf,
    n_iter=20,
    verbose=True
)

# Mantém na tabela o kernel com maior F1 médio
if resultado_svm_linear["f1_mean"] >= resultado_svm_rbf["f1_mean"]:
    resultado_svm = resultado_svm_linear
    busca_svm = busca_svm_linear
    resultado_svm["nome"] = "SVM (linear) — melhor"
else:
    resultado_svm = resultado_svm_rbf
    busca_svm = busca_svm_rbf
    resultado_svm["nome"] = "SVM (rbf) — melhor"

print(f"\nF1 SVM Linear: {resultado_svm_linear['f1_mean']:.4f}")
print(f"F1 SVM RBF:    {resultado_svm_rbf['f1_mean']:.4f}")
print(f"Escolhido: {resultado_svm['nome']}")

adicionar_resultado(resultado_svm)


### Melhor configuração do SVM


In [ ]:
print("Melhor resultado do SVM:")
print(resultado_svm)

print("\nMelhores hiperparâmetros encontrados:")
print(busca_svm.best_params_)

## Rede Neural MLP


In [ ]:
parametros_mlp = {
    "hidden_layer_sizes": [(16,), (32,), (64,), (32, 16), (64, 32)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.0005, 0.001, 0.01],
    "batch_size": [16, 32, 64]
}

resultado_mlp, busca_mlp = busca_aleatoria(
    nome_modelo="Rede Neural — MLP",
    estimador_ou_classe=MLPClassifier(
        random_state=SEED,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=20
    ),
    espaco_params=parametros_mlp,
    n_iter=30,
    verbose=True
)

adicionar_resultado(resultado_mlp)

### Melhor configuração da Rede Neural MLP


In [ ]:
print("Melhor resultado da Rede Neural / MLP:")
print(resultado_mlp)

print("\nMelhores hiperparâmetros encontrados:")
print(busca_mlp.best_params_)

## Comparação por validação cruzada no conjunto de treino

Reúno os resultados da validação de cinco folds e mantenho as referências históricas identificadas. Não considero esse ranking misto uma comparação controlada com a Entrega 2.

In [ ]:
tabela_final = exibir_tabela_comparacao()
tabela_final


In [ ]:
melhor_modelo_validacao = tabela_final.iloc[0]

print("Melhor modelo na validação cruzada pelo F1-score médio:")

if "Modelo" in tabela_final.columns:
    print(f"Modelo: {melhor_modelo_validacao['Modelo']}")
    print(f"F1 médio: {melhor_modelo_validacao['F1 médio']:.4f} (+/- {melhor_modelo_validacao['F1 std']:.4f})")
    print(f"Recall médio: {melhor_modelo_validacao['Recall médio']:.4f} (+/- {melhor_modelo_validacao['Recall std']:.4f})")
    print(f"Precisão média: {melhor_modelo_validacao['Precisão média']:.4f} (+/- {melhor_modelo_validacao['Precisão std']:.4f})")
    print(f"Acurácia média: {melhor_modelo_validacao['Acurácia média']:.4f} (+/- {melhor_modelo_validacao['Acurácia std']:.4f})")

else:
    print(f"Modelo: {melhor_modelo_validacao['nome']}")
    print(f"F1 médio: {melhor_modelo_validacao['f1_mean']:.4f} (+/- {melhor_modelo_validacao['f1_std']:.4f})")
    print(f"Recall médio: {melhor_modelo_validacao['recall_mean']:.4f} (+/- {melhor_modelo_validacao['recall_std']:.4f})")
    print(f"Precisão média: {melhor_modelo_validacao['precisao_mean']:.4f} (+/- {melhor_modelo_validacao['precisao_std']:.4f})")
    print(f"Acurácia média: {melhor_modelo_validacao['acuracia_mean']:.4f} (+/- {melhor_modelo_validacao['acuracia_std']:.4f})")

print("\nObservação:")
print(
    "Como a base é desbalanceada, os modelos devem ser analisados principalmente "
    "por F1-score e recall. A acurácia isolada pode ser enganosa, pois modelos que "
    "preveem majoritariamente a classe 'Sem UTI' podem atingir alta acurácia, mas "
    "falhar na identificação da classe minoritária."
)


In [ ]:
caminho_comparacao = PASTA_TABELAS / "comparacao_modelos.csv"
tabela_final.to_csv(caminho_comparacao, index=False)
print(f"Tabela final salva em {caminho_comparacao}")


## Avaliação final no conjunto reservado de teste

Avalio os modelos após a busca, sem reamostrar o teste. Uso o ranking para descrever esta execução; não o trato como validação independente de escolhas posteriores.

In [ ]:
RESULTADOS_TESTE_FINAL = []

# Baseline de classe majoritária
baseline_teste = DummyClassifier(strategy="most_frequent")
baseline_teste.fit(X_treino, y_treino)

avaliar_no_teste_final(
    nome_modelo="Baseline — classe majoritária",
    modelo_ou_busca=baseline_teste
)

# KNN com o pipeline oficial da Entrega 2
knn_entrega2 = construir_pipeline_oficial(
    KNeighborsClassifier(n_neighbors=5)
)

knn_entrega2.fit(X_treino, y_treino)

avaliar_no_teste_final(
    nome_modelo="KNN — pipeline Entrega 2",
    modelo_ou_busca=knn_entrega2
)

# Modelos avaliados na Entrega 3
avaliar_no_teste_final(
    nome_modelo="Regressão Logística",
    modelo_ou_busca=busca_logistica
)

avaliar_no_teste_final(
    nome_modelo="Naive Bayes",
    modelo_ou_busca=busca_naive_bayes
)

avaliar_no_teste_final(
    nome_modelo="Árvore de Decisão",
    modelo_ou_busca=busca_arvore
)

avaliar_no_teste_final(
    nome_modelo="SVM",
    modelo_ou_busca=busca_svm
)

avaliar_no_teste_final(
    nome_modelo="Rede Neural — MLP",
    modelo_ou_busca=busca_mlp
)

tabela_teste_final = exibir_tabela_teste_final()


## Gráficos comparativos




In [ ]:
tabela_plot_teste = tabela_teste_final.sort_values("F1 teste", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(tabela_plot_teste["Modelo"], tabela_plot_teste["F1 teste"])
ax.set_title("Comparação dos modelos no teste final — F1-score")
ax.set_xlabel("Modelo")
ax.set_ylabel("F1-score no teste final")
ax.set_ylim(0, max(tabela_plot_teste["F1 teste"].max() + 0.05, 0.1))

for i, valor in enumerate(tabela_plot_teste["F1 teste"]):
    ax.text(i, valor, f"{valor:.3f}", ha="center", va="bottom")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "f1_teste.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
tabela_recall_precisao = (
    tabela_teste_final
    .set_index("Modelo")[["Recall teste", "Precisão teste"]]
    .sort_values("Recall teste", ascending=False)
)

ax = tabela_recall_precisao.plot(kind="bar", figsize=(10, 5))
ax.set_title("Trade-off entre recall e precisão no teste final")
ax.set_xlabel("Modelo")
ax.set_ylabel("Valor da métrica")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
def padronizar_nome_modelo(nome):
    """Alinha nomes entre validação cruzada e teste final."""
    nome = str(nome)

    if "Baseline" in nome:
        return "Baseline"
    if "KNN" in nome:
        return "KNN — Entrega 2"
    if "Regressão Logística" in nome:
        return "Regressão Logística"
    if "Naive Bayes" in nome:
        return "Naive Bayes"
    if "Árvore de Decisão" in nome:
        return "Árvore de Decisão"
    if "SVM" in nome:
        return "SVM"
    if "Rede Neural" in nome or "MLP" in nome:
        return "Rede Neural — MLP"

    return nome

cv_f1 = tabela_final[["nome", "f1_mean"]].copy()
cv_f1["Modelo"] = cv_f1["nome"].apply(padronizar_nome_modelo)
cv_f1 = cv_f1[["Modelo", "f1_mean"]].rename(columns={"f1_mean": "F1 validação cruzada"})

teste_f1 = tabela_teste_final[["Modelo", "F1 teste"]].copy()
teste_f1["Modelo"] = teste_f1["Modelo"].apply(padronizar_nome_modelo)
teste_f1 = teste_f1.rename(columns={"F1 teste": "F1 teste final"})

comparacao_f1 = pd.merge(cv_f1, teste_f1, on="Modelo", how="inner")
comparacao_f1 = comparacao_f1.sort_values("F1 teste final", ascending=False)

display(comparacao_f1)

ax = comparacao_f1.set_index("Modelo").plot(kind="bar", figsize=(10, 5))
ax.set_title("F1-score: validação cruzada vs teste final")
ax.set_xlabel("Modelo")
ax.set_ylabel("F1-score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "f1_validacao_teste.png", dpi=150, bbox_inches="tight")
plt.show()


## Matrizes de confusão

Analiso a Árvore escolhida pela busca usando as previsões nos conjuntos reais de treino e teste. Não aplico SMOTE ao conjunto de teste.

In [ ]:
modelo_arvore_final = busca_arvore.best_estimator_

y_pred_arvore_treino = modelo_arvore_final.predict(X_treino)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_treino,
    y_pred_arvore_treino,
    display_labels=["Sem UTI", "Com UTI"],
    values_format="d",
    ax=ax,
    colorbar=False
)
ax.set_title("Matriz de confusão — Árvore de Decisão no treino")
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "matriz_confusao_treino.png", dpi=150, bbox_inches="tight")
plt.show()

y_pred_arvore_teste = modelo_arvore_final.predict(X_teste)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_teste,
    y_pred_arvore_teste,
    display_labels=["Sem UTI", "Com UTI"],
    values_format="d",
    ax=ax,
    colorbar=False
)
ax.set_title("Matriz de confusão — Árvore de Decisão no teste final")
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "matriz_confusao_teste.png", dpi=150, bbox_inches="tight")
plt.show()


## Conclusão

Obtive F1 de **0,2198** para a Árvore de Decisão, o maior valor observado no teste dessa execução. Identifiquei 10 dos 21 positivos e registrei 60 falsos positivos, com precisão de 0,1429 e recall de 0,4762.

No trabalho, obtivemos F1 de 0,1739 para SVM, 0,1579 para Regressão Logística, 0,1238 para MLP e 0,1176 para KNN e Naive Bayes. No baseline de classe majoritária, obtive F1 zero, apesar da acurácia de 0,9488.

Interpreto esses resultados considerando os poucos positivos, a baixa precisão e as regras de seleção da população. Predigo o registro de internação; não valido aqui a necessidade clínica de UTI.

Como próximos estudos, considero alternativas de imputação para indicadores, uma comparação comum de modelos e baselines e validação independente. Não utilizo o teste já consultado para escolher novas configurações.

In [ ]:
#!/usr/bin/env python3
"""
Exporta todos os dados necessários para o relatório da Entrega 3.
Cole este script no final do notebook e execute após todos os modelos rodarem.
Gera o arquivo local: results/reports/resumo_experimentos.json
"""

import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    confusion_matrix
)


# ─────────────────────────────────────────────────────────────
# HELPER: converte tipos numpy para Python nativo (json-safe)
# ─────────────────────────────────────────────────────────────
def _to_python(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_python(i) for i in obj]
    return obj


# ─────────────────────────────────────────────────────────────
# 1. DISTRIBUIÇÃO DA VARIÁVEL ALVO
# ─────────────────────────────────────────────────────────────
distrib_y = y.value_counts().sort_index()
total = len(y)

dados_distribuicao = {
    "total_registros": int(total),
    "sem_uti": {
        "quantidade": int(distrib_y.get(0, 0)),
        "percentual": round(distrib_y.get(0, 0) / total * 100, 2)
    },
    "com_uti": {
        "quantidade": int(distrib_y.get(1, 0)),
        "percentual": round(distrib_y.get(1, 0) / total * 100, 2)
    },
    "razao_desbalanceamento": round(distrib_y.get(0, 0) / max(distrib_y.get(1, 1), 1), 1)
}


# ─────────────────────────────────────────────────────────────
# 2. TAMANHO DOS CONJUNTOS TREINO / TESTE
# ─────────────────────────────────────────────────────────────
distrib_treino = y_treino.value_counts().sort_index()
distrib_teste  = y_teste.value_counts().sort_index()

dados_split = {
    "treino": {
        "total": int(len(y_treino)),
        "sem_uti": int(distrib_treino.get(0, 0)),
        "com_uti": int(distrib_treino.get(1, 0)),
        "pct_com_uti": round(distrib_treino.get(1, 0) / len(y_treino) * 100, 2)
    },
    "teste": {
        "total": int(len(y_teste)),
        "sem_uti": int(distrib_teste.get(0, 0)),
        "com_uti": int(distrib_teste.get(1, 0)),
        "pct_com_uti": round(distrib_teste.get(1, 0) / len(y_teste) * 100, 2)
    }
}


# ─────────────────────────────────────────────────────────────
# 3. TABELA DE VALIDAÇÃO CRUZADA (CV no treino)
# ─────────────────────────────────────────────────────────────
tabela_cv = exibir_tabela_comparacao()

colunas_cv = ["nome", "f1_mean", "f1_std", "recall_mean", "recall_std",
              "precisao_mean", "precisao_std", "acuracia_mean", "acuracia_std"]

colunas_presentes = [c for c in colunas_cv if c in tabela_cv.columns]

dados_cv = _to_python(
    tabela_cv[colunas_presentes]
    .sort_values("f1_mean", ascending=False)
    .reset_index(drop=True)
    .to_dict(orient="records")
)


# ─────────────────────────────────────────────────────────────
# 4. TABELA DE AVALIAÇÃO NO TESTE FINAL
# ─────────────────────────────────────────────────────────────
tabela_tf = pd.DataFrame(RESULTADOS_TESTE_FINAL)

dados_teste_final = _to_python(
    tabela_tf.sort_values("F1 teste", ascending=False)
    .reset_index(drop=True)
    .to_dict(orient="records")
)


# ─────────────────────────────────────────────────────────────
# 5. MELHORES HIPERPARÂMETROS POR MODELO
# ─────────────────────────────────────────────────────────────
dados_hiperparametros = {
    "Regressão Logística": _to_python(busca_logistica.best_params_),
    "Naive Bayes":         _to_python(busca_naive_bayes.best_params_),
    "Árvore de Decisão":   _to_python(busca_arvore.best_params_),
    "SVM":                 _to_python(busca_svm.best_params_),
    "Rede Neural MLP":     _to_python(busca_mlp.best_params_),
    "SVM kernel escolhido": resultado_svm["nome"]
}


# ─────────────────────────────────────────────────────────────
# 6. MATRIZES DE CONFUSÃO (no conjunto de teste)
# ─────────────────────────────────────────────────────────────
def _matriz(modelo_ou_busca, nome):
    m = modelo_ou_busca.best_estimator_ if hasattr(modelo_ou_busca, "best_estimator_") else modelo_ou_busca
    y_pred = m.predict(X_teste)
    cm = confusion_matrix(y_teste, y_pred)
    return {
        "modelo": nome,
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
    }

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier

baseline_final = DummyClassifier(strategy="most_frequent")
baseline_final.fit(X_treino, y_treino)

knn_e2 = construir_pipeline_oficial(KNeighborsClassifier(n_neighbors=5))
knn_e2.fit(X_treino, y_treino)

dados_matrizes = [
    _matriz(baseline_final,   "Baseline"),
    _matriz(knn_e2,           "KNN — Entrega 2"),
    _matriz(busca_logistica,  "Regressão Logística"),
    _matriz(busca_naive_bayes,"Naive Bayes"),
    _matriz(busca_arvore,     "Árvore de Decisão"),
    _matriz(busca_svm,        "SVM"),
    _matriz(busca_mlp,        "Rede Neural MLP"),
]


# ─────────────────────────────────────────────────────────────
# 7. COMPARAÇÃO SVM LINEAR vs RBF
# ─────────────────────────────────────────────────────────────
dados_svm_comparacao = {
    "linear": {
        "f1_mean":    resultado_svm_linear["f1_mean"],
        "f1_std":     resultado_svm_linear["f1_std"],
        "recall_mean":resultado_svm_linear["recall_mean"],
        "params":     _to_python(busca_svm_linear.best_params_)
    },
    "rbf": {
        "f1_mean":    resultado_svm_rbf["f1_mean"],
        "f1_std":     resultado_svm_rbf["f1_std"],
        "recall_mean":resultado_svm_rbf["recall_mean"],
        "params":     _to_python(busca_svm_rbf.best_params_)
    },
    "kernel_escolhido": resultado_svm["nome"]
}


# ─────────────────────────────────────────────────────────────
# 8. FEATURES SELECIONADAS PELO SelectKBest
#    (extraído do melhor pipeline da Árvore — mais estável)
# ─────────────────────────────────────────────────────────────
try:
    pipeline_arvore = busca_arvore.best_estimator_
    selector        = pipeline_arvore.named_steps["selector"]
    preprocessor    = pipeline_arvore.named_steps["preprocessor"]

    # nomes das colunas após o ColumnTransformer
    nomes_saida = preprocessor.get_feature_names_out()
    mask_selecionadas = selector.get_support()
    features_selecionadas_chi2 = [
        nomes_saida[i] for i, ok in enumerate(mask_selecionadas) if ok
    ]
    scores_chi2 = [
        round(float(selector.scores_[i]), 2) for i, ok in enumerate(mask_selecionadas) if ok
    ]
    dados_features = {
        "k": int(selector.k),
        "features": features_selecionadas_chi2,
        "scores_chi2": scores_chi2
    }
except Exception as e:
    dados_features = {"erro": str(e)}


# ─────────────────────────────────────────────────────────────
# 9. RESUMO EXECUTIVO — melhor modelo na CV e no teste
# ─────────────────────────────────────────────────────────────
melhor_cv    = max(dados_cv,          key=lambda x: x.get("f1_mean", 0))
melhor_teste = max(dados_teste_final, key=lambda x: x.get("F1 teste", 0))

dados_resumo = {
    "melhor_na_validacao_cruzada": {
        "nome":          melhor_cv.get("nome"),
        "f1_mean":       melhor_cv.get("f1_mean"),
        "f1_std":        melhor_cv.get("f1_std"),
        "recall_mean":   melhor_cv.get("recall_mean"),
        "precisao_mean": melhor_cv.get("precisao_mean"),
        "acuracia_mean": melhor_cv.get("acuracia_mean"),
    },
    "melhor_no_teste_final": {
        "nome":       melhor_teste.get("Modelo"),
        "f1_teste":   melhor_teste.get("F1 teste"),
        "recall":     melhor_teste.get("Recall teste"),
        "precisao":   melhor_teste.get("Precisão teste"),
        "acuracia":   melhor_teste.get("Acurácia teste"),
    }
}


# ─────────────────────────────────────────────────────────────
# 10. MONTAR E SALVAR
# ─────────────────────────────────────────────────────────────
payload = {
    "distribuicao_alvo":   dados_distribuicao,
    "split_treino_teste":  dados_split,
    "validacao_cruzada":   dados_cv,
    "teste_final":         dados_teste_final,
    "hiperparametros":     dados_hiperparametros,
    "matrizes_confusao":   dados_matrizes,
    "svm_comparacao":      dados_svm_comparacao,
    "features_chi2":       dados_features,
    "resumo_executivo":    dados_resumo,
}

CAMINHO_JSON = PASTA_RELATORIOS / "resumo_experimentos.json"

with open(CAMINHO_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("=" * 60)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 60)
print(f"Arquivo gerado: {CAMINHO_JSON}")
print()
print("─── Distribuição da variável alvo ───")
print(f"  Total: {dados_distribuicao['total_registros']} registros")
print(f"  Sem UTI: {dados_distribuicao['sem_uti']['quantidade']} ({dados_distribuicao['sem_uti']['percentual']}%)")
print(f"  Com UTI: {dados_distribuicao['com_uti']['quantidade']} ({dados_distribuicao['com_uti']['percentual']}%)")
print(f"  Razão: {dados_distribuicao['razao_desbalanceamento']}:1")
print()
print("─── Validação cruzada — ranking por F1 ───")
for r in dados_cv:
    nome = r.get("nome") or r.get("Modelo", "?")
    print(f"  {nome:<40} F1={r.get('f1_mean', 0):.4f} ± {r.get('f1_std', 0):.4f}")
print()
print("─── Teste final — ranking por F1 ───")
for r in dados_teste_final:
    print(f"  {r.get('Modelo', '?'):<40} F1={r.get('F1 teste', 0):.4f}")
print()
print("─── Melhor modelo ───")
print(f"  CV:    {dados_resumo['melhor_na_validacao_cruzada']['nome']}")
print(f"  Teste: {dados_resumo['melhor_no_teste_final']['nome']}")
print()
print(f"Use o arquivo '{CAMINHO_JSON}' para gerar o relatório final.")